# The convolutional synthetic: logs to trace

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_convolutional_synthetic.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: Course defaults. Edit the parameters cell and run again; every value is yours to change.

**Workflow**:
1. 2.4 A Synthetic from a Well Log
2. 1.2 The Wavelet Zoo
3. 2.3 Tuning and the Wedge

In [ ]:
# Ask Ogbon starter: Tie a well to surface seismic
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
"""A normal-incidence synthetic seismogram from a well log: the trace of a well tie.

Follows Sections 2.1 (From Impedance to Reflectivity), 2.2 (Convolution, the
Machine) and 2.4 (A Synthetic from a Well Log); the wavelets are Section 1.2's.
Units: velocity m/s, density g/cc, impedance m/s times g/cc, time ms (two-way).
Assumptions and limits:
- 1D convolution: normal incidence, primaries only, no transmission loss, no
  attenuation, no noise (Section 2.6, When Convolution Is Enough).
- The log is already in two-way time, one sample per DT_MS; a real log is in
  depth and needs a checkshot or an integrated sonic first.
- The wavelet is zero phase and known; a real tie estimates it from the data.
- The field: a well drilled where the field earth's gas column is thickest; its
  logs are read off the earth, and the recorded section it is tied to has its own
  constant-Q wavelet, noise and a datum shift, which the tie finds by correlation.

It prints the numbers Figure 12.5 computes for the same answers as name = value
lines; then it builds the field earth every starter shares, prints what it measures
there as [field] name = value lines, and draws one figure. Python 3.9+, numpy and
matplotlib; the earth's random numbers are seeded, so every run prints the same.
Edit the parameters and run again.
"""
import numpy as np
import matplotlib.pyplot as plt

# --- parameters from your interview ---
USE_SYNTHETIC_LOG = True  # False: you will paste your own log into main()
WAVELET = 'ricker'        # 'ricker' or 'ormsby'
F_HZ = 25.0               # Ricker peak frequency (Hz)
CORNERS_HZ = (5.0, 10.0, 40.0, 50.0)  # Ormsby corner frequencies, rising (Hz)
ORMSBY_TAPS = 121         # Ormsby length (samples)
DT_MS = 1.0               # sample interval (ms), the book's standard

In [ ]:
# --- the book's constants (not interview answers) ---
# the book's standard rocks (SSM.STANDARD, Section 3.3): P velocity (m/s), S velocity (m/s), density (g/cc)
ROCKS = {
    'shale': (2800, 1241.3793103448277, 2.2521031547808517),
    'brine sand': (2889.973997793982, 1504.4719169684909, 2.2545487499999997),
    'oil sand': (2543.1993596875604, 1529.5017037311773, 2.1813626422963988),
    'gas sand': (2341.92404704112, 1582.4208998550723, 2.0379042775100316)
}
# the stand-in log: (rock, samples) from the top down, one sample per DT_MS of two-way time
COLUMN = [('shale', 40), ('brine sand', 30), ('shale', 25), ('gas sand', 20), ('shale', 40), ('oil sand', 15), ('shale', 60)]
T_MAX_MS = 1000.0                # the field section's length (ms)
SEIS_F_HZ, SEIS_Q, SEIS_SNR = 25.0, 80.0, 5.0   # the recorded section: Ricker peak (Hz), Q and signal-to-noise
SEIS_SHIFT_MS = 12.0             # a datum shift in the recorded section, for the tie to find (ms)
TIE_WINDOW_MS, MAX_LAG_MS = (400.0, 900.0), 30.0   # the window the tie correlates, the largest shift tried (ms)

def report(name, value, fmt='%.6f'):
    """Print one result as name = value, the line Figure 12.5 checks against the book's kernel."""
    print(('%s = ' + fmt) % (name, value))

def style(ax, title, xlabel, ylabel, grid=True):
    """The starters' one look: a title, both axes labelled with their units, a light grid."""
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if grid:
        ax.grid(True, alpha=0.3)

MIN_HALF_MS = 40.0      # the book's tail rule (Figure 0.1): never shorter than 40 ms either side
TAIL_MS_HZ = 1200.0     # ... and 1.20 / f seconds, 1200 / f ms, either side

def ricker(f_hz, dt_ms, n):
    """The zero-phase Ricker of peak frequency f_hz (Hz) on n samples of dt_ms (ms), peak at the centre (Section 1.1)."""
    t = (np.arange(n) - n // 2) * dt_ms * 1e-3       # s
    a = (np.pi * f_hz * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def ricker_full(f_hz, dt_ms=1.0):
    """The Ricker taken whole by the book's tail rule: 81 taps at 30 Hz on 1 ms."""
    half = max(int(round(MIN_HALF_MS / dt_ms)), int(np.ceil(TAIL_MS_HZ / (f_hz * dt_ms))))
    return ricker(f_hz, dt_ms, 2 * half + 1)

def ormsby(f1, f2, f3, f4, dt_ms, n):
    """The zero-phase Ormsby wavelet of the trapezoid f1-f2-f3-f4 (Hz) on n samples (Section 1.2)."""
    t = (np.arange(n) - n // 2) * dt_ms * 1e-3       # s
    safe_t = np.where(t == 0, 1.0, t)
    def term(f):
        return np.pi * f * f * np.sinc(f * t) ** 2
    def edge(f):
        return np.where(t == 0, 2 * np.pi * f, np.sin(2 * np.pi * f * t) / safe_t)
    def skirt(fa, fb):
        if fb - fa <= 1e-9 * (1 + abs(fb)):
            return edge(0.5 * (fa + fb))
        return (term(fb) - term(fa)) / (fb - fa)
    return (skirt(f3, f4) - skirt(f1, f2)) / (np.pi * (f3 + f4 - f1 - f2))

# --- the Ogbon field earth: one seeded 2D earth that every starter shares ---
# In geological order: thin sand and shale beds with a regional dip, folded into an
# asymmetric anticline and cut by normal faults, the largest a listric growth fault
# (its throw grows with depth and its hanging wall rolls over into it); uplift, and an
# angular unconformity that truncates the crest; the reservoir sand subcrops beneath
# it, wedging out against it, shaling out to the left, sealed by a shale cap and
# holding gas above a flat contact; then a draped overburden with a channel. Each
# cell's velocity follows the
# compaction trend at its deepest burial (the rocks under the unconformity were once
# buried deeper), sands soft when shallow and hard when deep; Gardner's density; the
# mudrock and sandstone lines for S. Units: m, m/s, g/cc; times in ms.
FAULTS = [(0.66, 1.0, 1.4, 380.0), (0.80, -1.0, 0.6, 470.0), (0.22, -1.0, 0.5, 560.0)]   # (x / length, side, size, tip m)
COMPACTION = (1900.0, 1.4)                             # P velocity = 1900 m/s + 1.4 per metre of burial, before lithology
K_RES = 9                                              # the reservoir: bed 9 of the deep package, 70 m thick
SEAL_M = 30.0                                          # the shale cap on the reservoir's top: 30 m, and never under two cells

def field_earth(nx=240, nz=320, dx=15.0, dz=5.0, seed=7, crest=0.45, fold=1.0, dip=1.0, faults=None, throw=1.0,
                unconformity=True, seal=None, upper=None, lower=None, column_m=130.0):
    """The field earth on nz by nx cells of dz by dx (m). seal, upper and lower are (vp, vs, density) of the shale on the
    unconformity and of the reservoir above and below its flat contact, or None for the compaction trend's own rocks."""
    rng = np.random.default_rng(seed)
    x, z = np.arange(nx) * dx, np.arange(nz) * dz
    xn, zz, sc = x / x[-1], z[:, None] + 0.0 * x, nz * dz / 1600.0     # sc: the geometry scales with the grid's depth
    hump = np.exp(-0.5 * ((xn - crest) / np.where(xn < crest, 0.13, 0.25)) ** 2)   # the anticline, steep to the left
    swell = 1.0 + 0.15 * np.sin(2 * np.pi * (0.8 * xn + 0.37 * np.arange(26)[:, None]))   # each bed thickens and thins
    thick = sc * (24.0 + 22.0 * rng.random(26))[:, None] * swell
    thick[K_RES] = sc * 70.0
    beds = sc * 470.0 + np.cumsum(thick, axis=0) - thick + sc * (170.0 * dip * xn - 300.0 * fold * hump)
    if faults is not None:
        fs = [(rng.uniform(0.12, 0.88), rng.choice([-1.0, 1.0]), rng.uniform(0.4, 1.4), rng.uniform(300.0, 600.0))
              for k in range(faults)]
    fault = np.zeros((nz, nx), bool)
    for x0, side, size, tip in (FAULTS if faults is None else fs):
        tip, a = tip * sc, 0.2 / (1200.0 * sc) ** 1.3       # the plane: x / length = x0 + side a (depth - tip) to the 1.3, listric
        plane = np.where(side * (xn - x0) > 0, tip + (np.abs(xn - x0) / a) ** (1 / 1.3), -np.inf)   # its depth at each x
        for k in range(26):
            g = throw * size * sc * 90.0 * np.clip((beds[k] - tip) / (700.0 * sc), 0.0, 1.6)      # growth: deeper beds slip more
            xk = x0 + side * a * np.maximum(beds[k] - tip, 0.0) ** 1.3                          # where the plane cuts the bed
            g = g * (1.0 + 0.6 * np.exp(-np.abs(xn - xk) / 0.04))                                # the rollover into the fault
            beds[k] = beds[k] + np.clip(plane - beds[k], 0.0, g)       # the hanging wall drops; in the heave the bed lies on the plane
        ix = np.rint((x0 + side * a * np.maximum(z - tip, 0.0) ** 1.3) * (nx - 1)).astype(int)
        on = (z > tip) & (ix >= 0) & (ix < nx)
        fault[np.nonzero(on)[0], ix[on]] = True
    beds = np.maximum.accumulate(beds, axis=0)
    if unconformity:                                   # cut just under the reservoir's crest, so it subcrops on both flanks
        unc = beds[K_RES + 1, np.argmax(hump)] + sc * (15.0 + 160.0 * dip * (xn - crest)
                                                       + 10.0 * np.sin(2 * np.pi * (1.3 * xn + 0.2)))
    else:
        unc = beds[0] - sc * 30.0                      # no erosion: the overburden lies on the beds
    fault &= zz > unc                                  # the faults are older than the unconformity
    fr = np.cumsum(0.6 + rng.random(13))
    fr = (fr - fr[0]) / (fr[-1] - fr[0])
    over = [sc * 40.0 + (unc.min() - sc * 90.0) * f + 0.85 * f * f * (unc - unc.min()) - sc * 25.0 * f * hump for f in fr]
    tops = np.maximum.accumulate(np.vstack(over + list(np.maximum(beds, unc))), axis=0)
    lay = (zz[:, None, :] >= tops[None, :, :]).sum(axis=1)                    # layer 13 is the seal; bed k is layer 14 + k
    sand = np.where(lay < 13, rng.random(41)[lay] < 0.45, (lay - 14) % 2 == 1) & (lay > 0) & (lay != 13)
    chan = (zz >= over[5]) & (zz < over[5] + sc * 45.0 * np.exp(-0.5 * ((xn - 0.7) / 0.04) ** 2))
    res = lay == 14 + K_RES
    top = np.where(res, zz, np.inf).min(axis=0)
    cap = (zz < top) & (zz >= top - max(SEAL_M * sc, 2.0 * dz))   # the seal: shale on the sand's top, under a bed or the unconformity
    sand = np.where(cap, False, sand)
    burial = zz + np.where(lay >= 14, sc * 150.0 * hump, 0.0)                # uplift: the deep rocks were buried deeper
    vary = 1.0 + 0.035 * np.clip(rng.standard_normal(41), -1.2, 1.2)        # bed to bed variety
    vp = (COMPACTION[0] + COMPACTION[1] * burial) * vary[lay]
    vp = np.where(sand | chan, vp * (0.86 + 0.1 * np.minimum(burial / 1000.0, 1.8)), vp)
    vp = np.where(chan, vp * 0.92, vp)                 # the channel cuts a horizon and is filled with a soft sand
    sand |= chan
    vs = np.where(sand, 0.804 * vp - 856.0, (vp - 1360.0) / 1.16)            # the sandstone and mudrock lines (Castagna)
    den = 1.741 * (vp / 1000.0) ** 0.25                                      # Gardner
    ntg = np.clip((xn - 0.12) / 0.12, 0.0, 1.0)                              # the reservoir shales out to the left
    contact = float(top.min()) + column_m * sc         # the flat contact, column_m under the shallowest sand
    for rock, on in ((seal, (lay == 13) | cap), (upper, res & (zz < contact)), (lower, res & (zz >= contact))):
        if rock is not None:
            w = 1.0 if rock is seal else ntg
            vp, vs, den = [np.where(on, w * rock[i] + (1 - w) * q, q) for i, q in enumerate((vp, vs, den))]
    return dict(x=x, z=z, vp=vp, vs=vs, den=den, lay=lay, sand=sand, res=res, fault=fault, unc=unc, top=top,
                contact=contact, ntg=ntg, dx=dx, dz=dz, chan=chan, cap=cap, gas=res & (zz < contact) & (ntg > 0.5))

def field_time(m, dt_ms, t_max_ms):
    """Each trace from depth to two-way time by vertical rays: every cell array of m on a dt_ms grid to t_max_ms."""
    t_top = 2000.0 * np.cumsum(m['dz'] / m['vp'], axis=0) - 2000.0 * m['dz'] / m['vp']
    t = np.arange(0.0, t_max_ms, dt_ms)
    pick = np.array([np.searchsorted(t_top[:, i], t, 'right') - 1 for i in range(t_top.shape[1])]).T
    cols = np.arange(t_top.shape[1])[None, :]
    out = dict((k, m[k][pick, cols]) for k in ('vp', 'vs', 'den', 'lay', 'sand', 'res', 'gas'))
    seen = np.vstack([0 * cols, np.cumsum(m['fault'], axis=0)])                 # fault cells above each row
    after = np.maximum(np.vstack([pick[1:], pick[-1:] + 1]), pick + 1)
    out['fault'] = seen[np.minimum(after, seen.shape[0] - 1), cols] > seen[pick, cols]   # a fault cell inside the sample
    out['t'], out['t_top'] = t, t_top
    return out

def reflectivity(ai):
    """Normal-incidence coefficient at the top of every sample, down axis 0 (Section 2.1)."""
    rc = np.zeros_like(ai)
    rc[1:] = (ai[1:] - ai[:-1]) / (ai[1:] + ai[:-1])
    return rc

def q_peak_hz(f0, t_s, q):
    """Peak (Hz) of a Ricker of peak f0 after t_s seconds through constant Q: the root of 2 f f / (f0 f0) + (pi t / Q) f = 2."""
    b = np.pi * t_s / q
    return (-b + np.sqrt(b * b + 16.0 / f0 ** 2)) * f0 ** 2 / 4.0

def field_image(rc, dt_ms, f0, q=80.0, snr=None, seed=11):
    """Every trace convolved with a zero-phase Ricker whose peak falls with time as constant q says, plus coloured noise."""
    nt, dt = rc.shape[0], dt_ms * 1e-3
    f = np.fft.rfftfreq(2 * nt, dt)[:, None]
    def band(fp):                                      # the Ricker's spectrum (Section 1.1), over dt for a sum of samples
        return 2.0 / np.sqrt(np.pi) * f ** 2 / fp ** 3 * np.exp(-(f / fp) ** 2) / dt
    def filt(a, fp):
        return np.fft.irfft(np.fft.rfft(a, 2 * nt, axis=0) * band(fp), 2 * nt, axis=0)[:nt]
    fp = q_peak_hz(f0, np.arange(nt) * dt, q)
    knots = np.linspace(fp.min(), fp.max(), 4 if fp.max() - fp.min() > 1e-9 else 1)
    step = max((fp.max() - fp.min()) / 3.0, 1e-9)
    out = sum(np.clip(1.0 - np.abs(fp - k) / step, 0.0, 1.0)[:, None] * filt(rc, k) for k in knots)
    if snr:
        noise = filt(np.random.default_rng(seed).standard_normal(rc.shape), fp.mean())
        noise = (noise + np.roll(noise, 1, 1) + np.roll(noise, -1, 1)) / 3.0          # coherent over a few traces
        out = out + noise * np.sqrt((out ** 2).mean() / (noise ** 2).mean()) / snr
    return out

def fieldnote(name, value, fmt='%.3f'):
    """Print a number of the field earth as [field] name = value: computed here, not by Figure 12.5."""
    print(('[field] %s = ' + fmt) % (name, value))

def show(ax, a, x_m, y, title, ylabel='two-way time (ms)', clip=None):
    """A section as an image, black for positive, clipped at clip (or at its 99th percentile): km across, time or depth down."""
    c = clip if clip else np.percentile(np.abs(a), 99.0) + 1e-12
    ax.imshow(a, cmap='gray_r', vmin=-c, vmax=c, aspect='auto', extent=[x_m[0] / 1e3, x_m[-1] / 1e3, y[-1], y[0]])
    style(ax, title, 'distance (km)', ylabel, grid=False)

def validate(wavelet, f_hz, corners_hz, dt_ms):
    """Stop with a clear message on a parameter the model cannot use."""
    nyquist_hz = 500.0 / dt_ms
    if wavelet not in ('ricker', 'ormsby'):
        raise ValueError("WAVELET must be 'ricker' or 'ormsby', not %r" % (wavelet,))
    if not 0 < f_hz < nyquist_hz:
        raise ValueError('F_HZ must lie between 0 and the Nyquist frequency, %.0f Hz' % nyquist_hz)
    f1, f2, f3, f4 = corners_hz
    if not 0 <= f1 <= f2 < f3 <= f4 < nyquist_hz:
        raise ValueError('CORNERS_HZ must rise, f2 < f3, and stay under the Nyquist frequency, %.0f Hz' % nyquist_hz)

def stand_in_log(column, rocks):
    """P velocity (m/s) and density (g/cc), sample by sample, from (rock, samples) pairs."""
    vp_mps = np.concatenate([np.full(n, rocks[r][0]) for r, n in column])
    den_gcc = np.concatenate([np.full(n, rocks[r][2]) for r, n in column])
    return vp_mps, den_gcc

def convolve_centred(rc, w):
    """The convolutional trace (Section 2.2): the wavelet's centre on each reflection, as long as rc."""
    half = w.size // 2
    return np.convolve(rc, w)[half:half + rc.size]

def tie(syn, seis, t_ms, lags):
    """The correlation of the synthetic with the seismic trace inside the tie window, at each shift of the synthetic (samples)."""
    win = (t_ms >= TIE_WINDOW_MS[0]) & (t_ms < TIE_WINDOW_MS[1])
    def corr(k):
        s = np.roll(syn, k)[win]
        return float(np.dot(s, seis[win]) / np.sqrt(np.dot(s, s) * np.dot(seis[win], seis[win])))
    return np.array([corr(k) for k in lags])

def plot_tie(e, t, seis, iw, z, syn, k_best, cc0, cc1):
    """The impedance log, the tie at the well before and after the shift, and the synthetic in its section."""
    fig, ax = plt.subplots(1, 3, figsize=(12, 5), gridspec_kw={'width_ratios': [1, 1.2, 2.6]})
    tm = t['t']
    ax[0].plot(z / 1e3, tm, 'k', lw=0.8)
    style(ax[0], 'Impedance at the well', 'impedance (km/s x g/cc)', 'two-way time (ms)')
    g = 1.0 / np.abs(seis[:, iw]).max()
    ax[1].plot(seis[:, iw] * g, tm, color='0.5', lw=1.2, label='seismic at the well')
    ax[1].plot(syn * g, tm, 'r', lw=0.8, ls=':', label='synthetic, untied (r = %.2f)' % cc0)
    ax[1].plot(np.roll(syn, k_best) * g, tm, 'r', lw=0.9, label='shifted %+.0f ms (r = %.2f)' % (k_best * DT_MS, cc1))
    style(ax[1], 'The tie', 'amplitude (scaled)', '')
    ax[1].legend(fontsize=7, loc='lower left')
    show(ax[2], seis, e['x'], tm, 'The tied synthetic in its section, at the well')
    xw = e['x'][iw] / 1e3
    s = np.roll(syn, k_best) * 0.25 / np.abs(syn).max()
    ax[2].plot(xw + s, tm, 'r', lw=0.8)
    ax[2].fill_betweenx(tm, xw, xw + s, where=s > 0, color='r', lw=0)
    ax[2].axvline(xw, color='r', lw=0.4, ls='--')
    for a in ax:
        a.set_ylim(TIE_WINDOW_MS[1] + 100, TIE_WINDOW_MS[0] - 100)
    fig.suptitle('Well tie: the %s wavelet on the logs of a well drilled into the gas' % WAVELET)
    fig.tight_layout()
    plt.show()

def main():
    validate(WAVELET, F_HZ, CORNERS_HZ, DT_MS)
    if WAVELET == 'ricker':
        w = ricker_full(F_HZ, DT_MS)
    else:
        w = ormsby(*CORNERS_HZ, DT_MS, ORMSBY_TAPS)
    # 1. the book's stand-in column: the numbers Figure 12.5 computes
    vp_mps, den_gcc = stand_in_log(COLUMN, ROCKS)
    z = vp_mps * den_gcc                       # acoustic impedance (m/s x g/cc)
    rc = reflectivity(z)
    trace = convolve_centred(rc, w)
    k = int(np.abs(trace).argmax())
    report('samples', z.size, '%d')
    report('wavelet_taps', w.size, '%d')
    report('largest_abs_rc', np.abs(rc).max())
    report('trace_peak', trace[k])
    report('trace_peak_ms', k * DT_MS, '%.1f')
    report('trace_rms', np.sqrt((trace ** 2).mean()))
    # self-checks: one trace sample per log sample; positive impedances never reflect everything
    assert trace.size == z.size and np.abs(rc).max() < 1.0
    # 2. the field: a well drilled where the gas column is thickest; its logs, read off the earth in two-way time
    e = field_earth(seal=ROCKS['shale'], upper=ROCKS['gas sand'], lower=ROCKS['brine sand'])
    t = field_time(e, DT_MS, T_MAX_MS)
    iw = int(np.argmax(e['gas'].sum(axis=0)))
    vp_mps, den_gcc = t['vp'][:, iw], t['den'][:, iw]
    # YOUR LOG: replace the two arrays above with your sonic velocity (m/s) and density (g/cc), one value per DT_MS
    if not USE_SYNTHETIC_LOG:
        print('note: this run still uses the field earth\'s well; edit the two arrays above')
    z = vp_mps * den_gcc
    syn = convolve_centred(reflectivity(z), w)
    # the recorded section: the earth's constant-Q wavelet and noise, and a datum shift the tie must find
    seis = np.roll(field_image(reflectivity(t['vp'] * t['den']), DT_MS, SEIS_F_HZ, q=SEIS_Q, snr=SEIS_SNR), int(round(SEIS_SHIFT_MS / DT_MS)), axis=0)
    n = int(MAX_LAG_MS / DT_MS)
    lags = np.arange(-n, n + 1)                 # shifts of the synthetic, in samples
    cc = tie(syn, seis[:, iw], t['t'], lags)
    k_best = int(lags[np.argmax(cc)])
    fieldnote('well_x_m', e['x'][iw], '%.0f')
    fieldnote('gas_at_well_m', e['gas'][:, iw].sum() * e['dz'], '%.1f')
    fieldnote('datum_shift_ms', SEIS_SHIFT_MS, '%.1f')
    fieldnote('shift_found_ms', k_best * DT_MS, '%.1f')
    fieldnote('correlation_untied', cc[lags == 0][0])
    fieldnote('correlation_tied', cc.max())
    plot_tie(e, t, seis, iw, z, syn, k_best, cc[lags == 0][0], cc.max())

if __name__ == '__main__':
    main()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)